# Your Title Here

**Name(s)**: (La Li)

**Website Link**: (your website link)

In [53]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

from dsc80_utils import * # Feel free to uncomment and use this.

## Step 1: Introduction

In [54]:
# TODO
# Introduction and Question Identification


#### Introduction and Question Identification
Question chosen: Do tracks from artists with more followers tend to be more popular? Is artist size a better predictor of track popularity than audio features?

## Step 2: Data Cleaning and Exploratory Data Analysis


### Data Cleaning


In [55]:
# Load the two Spotify datasets.
tracks_raw = pd.read_csv("Spotify Music Tracks/Data/music_tracks.csv")
artists_raw = pd.read_csv("Spotify Music Tracks/Data/artists.csv")

tracks_raw.shape, artists_raw.shape


((114000, 22), (1162095, 5))

In [56]:
# Preview the original track-level data.
tracks_raw.head()


,Unnamed: 0,track_id,artists,album_name,...,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,...,0.71,87.92,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),...,0.27,77.49,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,...,0.12,76.33,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,...,0.14,181.74,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,...,0.17,NaN,4,acoustic


In [57]:
# Preview the original artist-level data.
artists_raw.head()


,id,followers,genres,name,popularity
0,0DheY5irMjBUeLybbCUEZ2,0.0,[],Armid & Amir Zare Pashai feat. Sara Rouzbehani,0
1,0DlhY15l3wsrnlfGio2bjU,5.0,[],ปูนา ภาวิณี,0
2,0DmRESX2JknGPQyO15yxg7,0.0,[],Sadaa,0
3,0DmhnbHjm1qw6NCYPeZNgJ,0.0,[],Tra'gruda,0
4,0Dn11fWM7vHQ3rinvWEl4E,2.0,[],Ioannis Panoutsopoulos,0


In [58]:
# Confirm that each Spotify genre has the same number of tracks.
genre_counts = tracks_raw.groupby("track_genre").size().sort_index()
genre_counts.to_dict()


{'acoustic': 1000,
 'afrobeat': 1000,
 'alt-rock': 1000,
 'alternative': 1000,
 'ambient': 1000,
 'anime': 1000,
 'black-metal': 1000,
 'bluegrass': 1000,
 'blues': 1000,
 'brazil': 1000,
 'breakbeat': 1000,
 'british': 1000,
 'cantopop': 1000,
 'chicago-house': 1000,
 'children': 1000,
 'chill': 1000,
 'classical': 1000,
 'club': 1000,
 'comedy': 1000,
 'country': 1000,
 'dance': 1000,
 'dancehall': 1000,
 'death-metal': 1000,
 'deep-house': 1000,
 'detroit-techno': 1000,
 'disco': 1000,
 'disney': 1000,
 'drum-and-bass': 1000,
 'dub': 1000,
 'dubstep': 1000,
 'edm': 1000,
 'electro': 1000,
 'electronic': 1000,
 'emo': 1000,
 'folk': 1000,
 'forro': 1000,
 'french': 1000,
 'funk': 1000,
 'garage': 1000,
 'german': 1000,
 'gospel': 1000,
 'goth': 1000,
 'grindcore': 1000,
 'groove': 1000,
 'grunge': 1000,
 'guitar': 1000,
 'happy': 1000,
 'hard-rock': 1000,
 'hardcore': 1000,
 'hardstyle': 1000,
 'heavy-metal': 1000,
 'hip-hop': 1000,
 'honky-tonk': 1000,
 'house': 1000,
 'idm': 1000,


#### Genre Selection

I selected classical, hip-hop, country, EDM, and jazz because they are musically distinct along several Spotify audio dimensions. Classical and jazz are often more acoustic and instrumental, EDM is typically more electronic, energetic, and danceable, hip-hop often has higher speechiness and explicit content, and country provides a contrasting popular acoustic genre. Each selected genre contains exactly 1000 tracks, so this choice avoids class-size imbalance while still allowing meaningful comparisons across popularity, artist size, and audio features.


In [59]:
# Keep five musically distinct genres with equal sample sizes.
selected_genres = ["classical", "hip-hop", "country", "edm", "jazz"]
tracks = tracks_raw[tracks_raw["track_genre"].isin(selected_genres)].copy()

tracks["track_genre"].value_counts().sort_index()


track_genre
classical    1000
country      1000
edm          1000
hip-hop      1000
jazz         1000
Name: count, dtype: int64

In [60]:
# Check missingness before adding new columns or joining artist metadata.
missing_before_cleaning = pd.DataFrame({
    "tracks_missing": tracks.isna().sum(),
}).query("tracks_missing > 0")

artists_missing = pd.DataFrame({
    "artists_missing": artists_raw.isna().sum(),
}).query("artists_missing > 0")

missing_before_cleaning, artists_missing


(       tracks_missing
 tempo            1196,
            artists_missing
 followers               11
 name                     3)

In [61]:
# Clean and engineer track-level columns.
# release_date appears as YYYY, YYYY-MM, or YYYY-MM-DD, so the first four characters give the year.
tracks = tracks.drop(columns=["Unnamed: 0"], errors="ignore")
tracks["release_year"] = pd.to_numeric(tracks["release_date"].str[:4], errors="coerce")
tracks["decade"] = (tracks["release_year"] // 10 * 10).astype("Int64")

# Multiple artists are separated by semicolons. Use the first listed artist as the main artist for joining.
tracks["main_artist"] = tracks["artists"].str.split(";").str[0]

# popularity == 0 is valid according to the data dictionary, but it is suspiciously common.
# Keep it as a real value, and create flags for analysis instead of replacing it with NaN.
tracks["popularity_zero"] = tracks["popularity"].eq(0)
tracks["popular_70"] = tracks["popularity"].ge(70)


In [62]:
# Some artist names appear more than once in artists.csv.
# Keep the row with the highest follower count so each artist name maps to at most one row.
artists_dedup = (
    artists_raw
    .sort_values("followers", ascending=False)
    .drop_duplicates("name")
)

merged = tracks.merge(
    artists_dedup,
    left_on="main_artist",
    right_on="name",
    how="left",
    suffixes=("_track", "_artist")
)

# followers are highly right-skewed, so use a log transform for artist-size analysis.
merged["log_followers"] = np.log1p(merged["followers"])
merged["missing_artist_match"] = merged["followers"].isna()

tracks.shape, merged.shape


((5000, 26), (5000, 33))

In [63]:
# Check important data-quality issues after cleaning.
cleaning_checks = pd.Series({
    "selected_track_rows": tracks.shape[0],
    "merged_rows": merged.shape[0],
    "missing_tempo_rows": tracks["tempo"].isna().sum(),
    "missing_artist_followers_after_join": merged["followers"].isna().sum(),
    "duplicated_track_id_rows": tracks["track_id"].duplicated().sum(),
    "track_ids_with_multiple_selected_genres": (tracks.groupby("track_id")["track_genre"].nunique() > 1).sum(),
    "popularity_zero_rows": tracks["popularity_zero"].sum(),
})

cleaning_checks


selected_track_rows                        5000
merged_rows                                5000
missing_tempo_rows                         1196
missing_artist_followers_after_join          45
duplicated_track_id_rows                    100
track_ids_with_multiple_selected_genres      16
popularity_zero_rows                       2317
dtype: int64

In [64]:
# Examine popularity == 0 in the full dataset and in the selected genres.
popularity_zero_summary = pd.DataFrame({
    "scope": ["full dataset", "selected genres"],
    "rows": [len(tracks_raw), len(tracks)],
    "zero_popularity_rows": [
        (tracks_raw["popularity"] == 0).sum(),
        tracks["popularity_zero"].sum()
    ],
    "zero_popularity_rate": [
        (tracks_raw["popularity"] == 0).mean(),
        tracks["popularity_zero"].mean()
    ]
})

popularity_zero_summary


,scope,rows,zero_popularity_rows,zero_popularity_rate
0,full dataset,114000,16020,0.14
1,selected genres,5000,2317,0.46


In [65]:
# Show examples where track popularity is 0 despite a large artist following.
zero_popular_artist_examples = (
    merged[merged["popularity_zero"]]
    .sort_values(["followers", "popularity_artist"], ascending=False)
    [["track_name", "main_artist", "track_genre", "popularity_track", "followers", "popularity_artist"]]
    .head(10)
)

zero_popular_artist_examples


,track_name,main_artist,track_genre,popularity_track,followers,popularity_artist
2070,Leave Before You Love Me,Marshmello,edm,0,3.02e+07,88.0
2088,Happier,Marshmello,edm,0,3.02e+07,88.0
2119,Leave Before You Love Me,Marshmello,edm,0,3.02e+07,88.0
...,...,...,...,...,...,...
3179,X ÚLTIMA VEZ,Daddy Yankee,hip-hop,0,2.28e+07,91.0
3184,X ÚLTIMA VEZ,Daddy Yankee,hip-hop,0,2.28e+07,91.0
3195,RUMBATÓN,Daddy Yankee,hip-hop,0,2.28e+07,91.0


In [66]:
# This is the cleaned DataFrame head to use in the report website.
cleaned_preview_cols = [
    "track_id", "track_name", "main_artist", "track_genre", "popularity_track",
    "popularity_zero", "popular_70", "danceability", "energy", "valence",
    "tempo", "release_year", "decade", "followers", "log_followers",
    "popularity_artist"
]

merged[cleaned_preview_cols].head()


,track_id,track_name,main_artist,track_genre,...,decade,followers,log_followers,popularity_artist
0,7wrYBASu0OoxoDErd4Edxd,Zara Zara,Bombay Jayashri,classical,...,2000,124188.0,11.73,55.0
1,72HdutlIHBZJ7WT1xVAAZT,Kajra Re,Shankar,classical,...,2000,11604.0,9.36,45.0
2,7JGgKHHDgJCJkQCQxyHHdl,Zara Zara - Lofi,Bombay Jayashri,classical,...,1980,124188.0,11.73,55.0
3,3YRj4jmwois2ctPnhwSwFo,Vaseegara,Bombay Jayashri,classical,...,1970,124188.0,11.73,55.0
4,3tp3ij9dtY3CacQgd1OvRf,Zara Zara - LoFi Chill,Bombay Jayashri,classical,...,1980,124188.0,11.73,55.0


#### Data Cleaning Summary

I filtered the track dataset to five musically distinct genres: classical, hip-hop, country, EDM, and jazz. Each selected genre has exactly 1000 tracks, giving 5000 track rows before joining artist information. I extracted `release_year` and `decade` from the mixed-format `release_date` column, created `main_artist` from the first semicolon-separated artist name, and joined the track data to `artists.csv` using that main artist name.

Because `artists.csv` contains repeated artist names, I first sorted artists by `followers` and kept the highest-follower row for each name before joining. This prevents the merge from duplicating tracks; the cleaned merged dataset still has 5000 rows. After the join, 45 tracks are missing artist follower information, and `tempo` has 1196 missing values in the selected tracks.

A major data-quality issue is that many tracks have `popularity == 0`. In the full dataset, 16020 out of 114000 tracks have popularity 0, and this issue is even stronger in the selected genres. Some of these zero-popularity tracks are by very popular artists, so I do not interpret every 0 as literally meaning no popularity. However, because the data dictionary defines popularity as a valid 0-100 score and does not say that 0 means missing, I keep these values rather than replacing them with `NaN`. To make this limitation explicit, I created `popularity_zero` and `popular_70` indicator columns. The first tracks suspicious zero scores, while the second supports a more robust binary popularity target later in the project.

I also found 100 duplicated `track_id` rows in the selected data, with 16 track IDs appearing under more than one selected genre. I kept these rows because the dataset is organized as genre-labeled track observations, but I will avoid train-test leakage from duplicated tracks during prediction.


### Univariate Analysis


In [67]:
# Distribution of the main outcome, track popularity, across selected genres.
fig_popularity = px.histogram(
    merged,
    x="popularity_track",
    color="track_genre",
    marginal="box",
    nbins=40,
    barmode="overlay",
    opacity=0.55,
    title="Distribution of Track Popularity Across Selected Genres",
    labels={
        "popularity_track": "Track Popularity",
        "track_genre": "Genre"
    }
)
fig_popularity.show()


In [68]:
# Distribution of artist size after a log transform.
fig_followers = px.histogram(
    merged.dropna(subset=["log_followers"]),
    x="log_followers",
    color="track_genre",
    marginal="box",
    nbins=35,
    barmode="overlay",
    opacity=0.55,
    title="Distribution of Log Artist Followers Across Selected Genres",
    labels={
        "log_followers": "Log Artist Followers",
        "track_genre": "Genre"
    }
)
fig_followers.show()


In [69]:
# Distribution of one audio feature to show that the selected genres are musically distinct.
fig_energy = px.violin(
    merged,
    x="track_genre",
    y="energy",
    color="track_genre",
    box=True,
    points=False,
    title="Distribution of Energy by Genre",
    labels={
        "track_genre": "Genre",
        "energy": "Energy"
    }
)
fig_energy.show()


The popularity distribution has a large spike at 0, especially for jazz and country, which supports treating zero popularity as an important data-quality limitation rather than ignoring it. The artist follower distribution is strongly right-skewed even after the log transform, showing why raw follower counts would be difficult to compare directly. The energy plot confirms that the selected genres differ substantially in audio characteristics, with EDM generally more energetic than classical and jazz.


### Bivariate Analysis


In [70]:
# Compare track popularity across artist follower quintiles.
# This is more interpretable than a single linear trend line because popularity has many zeros.
bivar_followers = merged.dropna(subset=["log_followers"]).copy()
bivar_followers["follower_group"] = pd.qcut(
    bivar_followers["log_followers"],
    q=5,
    labels=["Lowest", "Low", "Middle", "High", "Highest"],
    duplicates="drop"
)

fig_pop_by_followers = px.box(
    bivar_followers,
    x="follower_group",
    y="popularity_track",
    color="follower_group",
    title="Track Popularity by Artist Follower Group",
    labels={
        "follower_group": "Artist Follower Group",
        "popularity_track": "Track Popularity"
    }
)
fig_pop_by_followers.show()


In [71]:
# Compare popularity against an audio feature.
fig_energy_popularity = px.scatter(
    merged,
    x="energy",
    y="popularity_track",
    color="track_genre",
    opacity=0.45,
    title="Track Popularity vs. Energy by Genre",
    labels={
        "energy": "Energy",
        "popularity_track": "Track Popularity",
        "track_genre": "Genre"
    },
    hover_data=["track_name", "main_artist"]
)
fig_energy_popularity.update_traces(marker={"size": 5})
fig_energy_popularity.show()


In [72]:
# Use the binary popularity threshold to reduce the influence of suspicious zero-popularity values.
popular_rate_by_followers = (
    bivar_followers
    .groupby("follower_group", observed=True)
    .agg(
        n_tracks=("track_id", "size"),
        mean_popularity=("popularity_track", "mean"),
        median_popularity=("popularity_track", "median"),
        zero_popularity_rate=("popularity_zero", "mean"),
        popular_70_rate=("popular_70", "mean")
    )
    .reset_index()
)

fig_popular_rate = px.bar(
    popular_rate_by_followers,
    x="follower_group",
    y="popular_70_rate",
    text="popular_70_rate",
    title="Share of Popular Tracks by Artist Follower Group",
    labels={
        "follower_group": "Artist Follower Group",
        "popular_70_rate": "Proportion with Popularity >= 70"
    }
)
fig_popular_rate.update_traces(texttemplate="%{text:.1%}", textposition="outside")
fig_popular_rate.update_yaxes(tickformat=".0%", range=[0, popular_rate_by_followers["popular_70_rate"].max() * 1.25])
fig_popular_rate.show()


The relationship between artist followers and track popularity is not cleanly linear because many tracks have popularity 0. Grouping artists into follower quintiles gives a clearer view: the highest-follower group has a noticeably higher share of tracks with popularity at least 70, even though zero-popularity tracks still appear in every follower group. The energy scatter plot shows that audio features alone do not explain popularity cleanly; genre and artist size both appear to matter.


### Interesting Aggregates


In [73]:
# Aggregate key popularity, missingness, artist-size, and audio-feature statistics by genre.
genre_summary = (
    merged
    .groupby("track_genre")
    .agg(
        n_tracks=("track_id", "size"),
        mean_popularity=("popularity_track", "mean"),
        median_popularity=("popularity_track", "median"),
        zero_popularity_rate=("popularity_zero", "mean"),
        popular_70_rate=("popular_70", "mean"),
        tempo_missing_rate=("tempo", lambda s: s.isna().mean()),
        median_log_followers=("log_followers", "median"),
        mean_energy=("energy", "mean"),
        mean_danceability=("danceability", "mean"),
        mean_valence=("valence", "mean")
    )
    .sort_values("popular_70_rate", ascending=False)
)

genre_summary


,n_tracks,mean_popularity,median_popularity,zero_popularity_rate,...,median_log_followers,mean_energy,mean_danceability,mean_valence
track_genre,,,,,,,,,
edm,1000,35.03,47.0,0.36,...,14.53,0.76,0.65,0.47
hip-hop,1000,37.76,58.0,0.29,...,15.08,0.68,0.74,0.55
country,1000,17.03,0.0,0.59,...,13.71,0.60,0.56,0.52
jazz,1000,13.63,0.0,0.68,...,13.89,0.35,0.51,0.49
classical,1000,13.05,3.0,0.40,...,13.78,0.19,0.38,0.38


In [74]:
# A compact version of the aggregate table for the website.
website_genre_summary = genre_summary.assign(
    zero_popularity_rate=lambda df: (df["zero_popularity_rate"] * 100).round(1),
    popular_70_rate=lambda df: (df["popular_70_rate"] * 100).round(1),
    tempo_missing_rate=lambda df: (df["tempo_missing_rate"] * 100).round(1),
    mean_popularity=lambda df: df["mean_popularity"].round(1),
    median_popularity=lambda df: df["median_popularity"].round(1),
    median_log_followers=lambda df: df["median_log_followers"].round(2),
    mean_energy=lambda df: df["mean_energy"].round(2),
    mean_danceability=lambda df: df["mean_danceability"].round(2),
    mean_valence=lambda df: df["mean_valence"].round(2),
).rename(columns={
    "zero_popularity_rate": "zero_popularity_rate_pct",
    "popular_70_rate": "popular_70_rate_pct",
    "tempo_missing_rate": "tempo_missing_rate_pct"
})

website_genre_summary


,n_tracks,mean_popularity,median_popularity,zero_popularity_rate_pct,...,median_log_followers,mean_energy,mean_danceability,mean_valence
track_genre,,,,,,,,,
edm,1000,35.0,47.0,36.2,...,14.53,0.76,0.65,0.47
hip-hop,1000,37.8,58.0,28.9,...,15.08,0.68,0.74,0.55
country,1000,17.0,0.0,58.7,...,13.71,0.60,0.56,0.52
jazz,1000,13.6,0.0,68.1,...,13.89,0.35,0.51,0.49
classical,1000,13.1,3.0,39.8,...,13.78,0.19,0.38,0.38


In [75]:
# Markdown table source for the website report, without relying on optional tabulate.
def df_to_markdown(df):
    table = df.reset_index()
    headers = [str(col) for col in table.columns]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |"
    ]
    for row in table.astype(str).itertuples(index=False):
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines)

print(df_to_markdown(website_genre_summary))


| track_genre | n_tracks | mean_popularity | median_popularity | zero_popularity_rate_pct | popular_70_rate_pct | tempo_missing_rate_pct | median_log_followers | mean_energy | mean_danceability | mean_valence |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| edm | 1000 | 35.0 | 47.0 | 36.2 | 18.2 | 14.1 | 14.53 | 0.76 | 0.65 | 0.47 |
| hip-hop | 1000 | 37.8 | 58.0 | 28.9 | 13.4 | 17.6 | 15.08 | 0.68 | 0.74 | 0.55 |
| country | 1000 | 17.0 | 0.0 | 58.7 | 8.7 | 21.8 | 13.71 | 0.6 | 0.56 | 0.52 |
| jazz | 1000 | 13.6 | 0.0 | 68.1 | 2.5 | 31.2 | 13.89 | 0.35 | 0.51 | 0.49 |
| classical | 1000 | 13.1 | 3.0 | 39.8 | 0.7 | 34.9 | 13.78 | 0.19 | 0.38 | 0.38 |


The grouped table shows that popularity differs strongly by genre. EDM and hip-hop have the highest shares of tracks with popularity at least 70, while jazz and classical have much lower popular-track rates. The table also shows why `popularity == 0` needs to be discussed explicitly: jazz and country have especially high zero-popularity rates, which could distort analyses based only on mean popularity.


## Step 3: Assessment of Missingness

In [18]:
# TODO

## Step 4: Hypothesis Testing

In [19]:
# TODO

## Step 5: Framing a Prediction Problem

In [20]:
# TODO

## Step 6: Baseline Model

In [21]:
# TODO

## Step 7: Final Model

In [22]:
# TODO

## Step 8: Fairness Analysis

In [23]:
# TODO